# Scraping Respiratory Virus Detections Data

I will now scrape the respiratory virus detections data from the Canadian public health website. The goal is to extract 'Table 2' and convert it into a pandas DataFrame.

<a href="https://colab.research.google.com/github/techseeko/AAI614_Haidar/blob/main/Week-3/Saoud_VirusDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import requests # Keeping import for now, but not used in new logic
from bs4 import BeautifulSoup
import pandas as pd
import asyncio
from playwright.async_api import async_playwright

In [10]:
# Install Playwright Python package
print("Installing playwright Python package...")
!pip install playwright > /dev/null

Installing playwright Python package...


In [11]:
# --- Playwright setup and scraping logic ---

# Install Playwright system dependencies and browser binaries directly in this cell
# This ensures that Playwright's browser can launch correctly.
print("Updating apt-get and installing essential system dependencies...")
!apt-get update > /dev/null
!apt-get install -y libnss3 libxss1 libatk1.0-0 libgtk-3-0 libgdk-pixbuf2.0-0 libxcomposite1 libgbm-dev libcups2 > /dev/null # Essential common dependencies
print("Installing additional Playwright system dependencies...")
!playwright install-deps # Let this output be visible in case of errors
print("Installing Playwright browser binaries...")
!playwright install > /dev/null # Suppress verbose output

Updating apt-get and installing essential system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Installing additional Playwright system dependencies...
Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'mai

In [12]:
url = 'https://www.canada.ca/en/public-health/services/surveillance/respiratory-virus-detections-canada/2018-2019/respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html'

async def get_page_content(url_to_scrape):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        try:
            # Increased timeout for page load in Playwright to 90 seconds
            await page.goto(url_to_scrape, timeout=90000)
            content = await page.content()
            return content
        except Exception as e:
            print(f"Playwright failed to load page: {e}")
            return None
        finally:
            await browser.close()

# Use top-level await for Colab environment
try:
    page_content = await get_page_content(url)

    if page_content:
        print("Successfully fetched page content using Playwright.")
        soup = BeautifulSoup(page_content, 'html.parser')

        # Find 'Table 2'. Based on inspection, it might be the second table on the page.
        # More robust way: find the heading 'Table 2' and then find the next table sibling.
        table_heading = soup.find('h3', string=lambda text: text and 'Table 2' in text)
        if table_heading:
            table = table_heading.find_next_sibling('table')
        else:
            # Fallback if the heading isn't found, try to get the second table
            tables = soup.find_all('table')
            if len(tables) > 1:
                table = tables[1] # Assuming 'Table 2' is the second table
            else:
                table = None
                print("Could not find 'Table 2' on the page.")

        if table:
            headers = [th.text.strip() for th in table.find('thead').find_all('th')]
            data = []
            for row in table.find('tbody').find_all('tr'):
                cells = row.find_all('td')
                # Exclude rows that are just headers or empty
                if cells:
                    data.append([cell.text.strip() for cell in cells])

            df = pd.DataFrame(data, columns=headers)
            print("DataFrame created successfully from Table 2 using Playwright.")
            display(df.head())
            df.info()
        else:
            print("No table found for extraction after Playwright fetch.")
    else:
        print("Failed to retrieve page content using Playwright.")

except Exception as e:
    print(f"An error occurred during Playwright scraping: {e}")

Successfully fetched page content using Playwright.
DataFrame created successfully from Table 2 using Playwright.


,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
4,Atlantic,6741,81,1,880,962,4,6747,272,2859,...,33,0,2854,101,2854,18,2854,475,1555,7


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Reporting Laboratory   41 non-null     object
 1   Flu Tested             41 non-null     object
 2   A(H1N1)pdm09 Positive  41 non-null     object
 3   A(H3) Positive         41 non-null     object
 4   A(UnS) Positive        41 non-null     object
 5   Total Flu A Positive   41 non-null     object
 6   Total Flu B Positive   41 non-null     object
 7   RSV Tested             41 non-null     object
 8   RSV Positive           41 non-null     object
 9   PIV Tested             41 non-null     object
 10  PIV 1 Positive         41 non-null     object
 11  PIV 2 Positive         41 non-null     object
 12  PIV 3 Positive         41 non-null     object
 13  PIV 4 Positive         41 non-null     object
 14  Other PIV Positive     41 non-null     object
 15  Adeno Tested           41

To create a DataFrame with one row for each province or territory, I'll filter the `df` DataFrame to exclude aggregate rows like 'National' or 'Total'. I'll define a list of Canadian provinces and territories and then use this list to select the relevant rows.

In [13]:
canadian_provinces_territories = [
    'Newfoundland',
    'Prince Edward Island',
    'Nova Scotia',
    'New Brunswick',
    'Quebec',
    'Ontario',
    'Manitoba',
    'Saskatchewan',
    'Alberta',
    'British Columbia',
    'Yukon',
    'Northwest Territories',
    'Nunavut'
]

# Filter the DataFrame to include only rows corresponding to provinces/territories
df_provinces = df[df['Reporting Laboratory'].isin(canadian_provinces_territories)].copy()

print("DataFrame with one row per province/territory:")
display(df_provinces.head())
df_provinces.info()

DataFrame with one row per province/territory:


,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
29,Manitoba,4911,192,3,486,681,3,4828,101,2254,...,10,0,2254,25,1214,9,2254,310,1248,12


<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 0 to 38
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Reporting Laboratory   7 non-null      object
 1   Flu Tested             7 non-null      object
 2   A(H1N1)pdm09 Positive  7 non-null      object
 3   A(H3) Positive         7 non-null      object
 4   A(UnS) Positive        7 non-null      object
 5   Total Flu A Positive   7 non-null      object
 6   Total Flu B Positive   7 non-null      object
 7   RSV Tested             7 non-null      object
 8   RSV Positive           7 non-null      object
 9   PIV Tested             7 non-null      object
 10  PIV 1 Positive         7 non-null      object
 11  PIV 2 Positive         7 non-null      object
 12  PIV 3 Positive         7 non-null      object
 13  PIV 4 Positive         7 non-null      object
 14  Other PIV Positive     7 non-null      object
 15  Adeno Tested           7 non-nu

It appears that only 7 out of the 13 provinces/territories were found. Let's examine the unique values in the 'Reporting Laboratory' column of the original `df` to understand why some were not included. We'll also identify which provinces/territories from our list were not present in the data.

In [14]:
print("Unique values in 'Reporting Laboratory' column of original df:")
unique_reporting_labs = df['Reporting Laboratory'].unique()
for lab in unique_reporting_labs:
    print(f"- {lab}")

# Identify provinces/territories from our list that were not found in df
not_found_provinces = [p for p in canadian_provinces_territories if p not in unique_reporting_labs]

if not_found_provinces:
    print("\nThe following provinces/territories from your list were not found in the original DataFrame:")
    for p in not_found_provinces:
        print(f"- {p}")
else:
    print("\nAll listed provinces/territories were found in the original DataFrame, which is unexpected given the row count discrepancy.")

Unique values in 'Reporting Laboratory' column of original df:
- Newfoundland
- Prince Edward Island
- Nova Scotia
- New Brunswick
- Atlantic
- Région    Nord-Est
- Québec-Chaudière-Appalaches
- Centre-du-Québec
- Montréal-Laval
- Ouest du Québec
- Montérégie
- Province of Québec
- Ottawa P.H.L.
- CHEO - Ottawa
- Kingston P.H.L.
- UHN / Mount Sinai Hospital
- P.H.O.L. - Toronto
- Sick Kids Hospital - Toronto
- Sunnybrook & Women's    College HSC
- Sault Ste. Marie P.H.L.
- Timmins P.H.L.
- St. Joseph's - London
- London P.H.L.
- Orillia P.H.L.
- Thunder Bay P.H.L.
- Sudbury P.H.L.
- Hamilton P.H.L.
- Peterborough    P.H.L.
- Province of Ontario
- Manitoba
- Regina
- Saskatoon
- Province of    Saskatchewan
- Province of Alberta
- Prairies
- British    Columbia
- Yukon
- Northwest    Territories
- Nunavut
- Territories
- CANADA

The following provinces/territories from your list were not found in the original DataFrame:
- Quebec
- Ontario
- Saskatchewan
- Alberta
- British Columbia
- Nor

It appears the 'Reporting Laboratory' column contains variations like 'Province of Québec' instead of just 'Quebec', and some entries have extra spaces. To fix this, I will clean the 'Reporting Laboratory' column in the original `df` by removing the 'Province of ' prefix and stripping any leading/trailing whitespace. Then, I will re-filter to correctly extract all 13 provinces and territories.

In [16]:
# Clean the 'Reporting Laboratory' column in the original df
df['Reporting Laboratory'] = df['Reporting Laboratory'].astype(str)
# Remove 'Province of ' prefix
df['Reporting Laboratory'] = df['Reporting Laboratory'].str.replace('Province of ', '', regex=False)
# Normalize whitespace (replace multiple spaces with single space) and strip leading/trailing spaces
df['Reporting Laboratory'] = df['Reporting Laboratory'].str.replace(r'\s+', ' ', regex=True).str.strip()
# Handle specific name variations, e.g., 'Québec' to 'Quebec'
df['Reporting Laboratory'] = df['Reporting Laboratory'].str.replace('Québec', 'Quebec', regex=False)

# Define the list of Canadian provinces and territories (ensure names match the cleaned data)
canadian_provinces_territories = [
    'Newfoundland',
    'Prince Edward Island',
    'Nova Scotia',
    'New Brunswick',
    'Quebec',
    'Ontario',
    'Manitoba',
    'Saskatchewan',
    'Alberta',
    'British Columbia',
    'Yukon',
    'Northwest Territories',
    'Nunavut'
]

# Re-filter the DataFrame to include only rows corresponding to provinces/territories
df_provinces = df[df['Reporting Laboratory'].isin(canadian_provinces_territories)].copy()

print("DataFrame with one row per province/territory after cleaning:")
display(df_provinces.head())
df_provinces.info()

# Verify if all 13 provinces/territories are now present
print(f"\nNumber of provinces/territories found: {len(df_provinces)}")
missing_after_cleaning = [p for p in canadian_provinces_territories if p not in df_provinces['Reporting Laboratory'].unique()]
if missing_after_cleaning:
    print(f"Still missing: {missing_after_cleaning}")
else:
    print("All listed provinces/territories are now present in the df_provinces DataFrame.")

DataFrame with one row per province/territory after cleaning:


,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
11,Quebec,38395,0,0,5974,5975,81,35755,2816,11201,...,36,0,11288,373,10786,63,N.A.,N.A.,10703,310


<class 'pandas.core.frame.DataFrame'>
Index: 13 entries, 0 to 38
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Reporting Laboratory   13 non-null     object
 1   Flu Tested             13 non-null     object
 2   A(H1N1)pdm09 Positive  13 non-null     object
 3   A(H3) Positive         13 non-null     object
 4   A(UnS) Positive        13 non-null     object
 5   Total Flu A Positive   13 non-null     object
 6   Total Flu B Positive   13 non-null     object
 7   RSV Tested             13 non-null     object
 8   RSV Positive           13 non-null     object
 9   PIV Tested             13 non-null     object
 10  PIV 1 Positive         13 non-null     object
 11  PIV 2 Positive         13 non-null     object
 12  PIV 3 Positive         13 non-null     object
 13  PIV 4 Positive         13 non-null     object
 14  Other PIV Positive     13 non-null     object
 15  Adeno Tested           13 non-